In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
import os


device = torch.device("cuda")

In [ ]:
class Config:
    T = 5
    IMG_SIZE = (512, 512)
    BATCH_SIZE = 8

    LR_INIT = 0.01
    LR_DECAY = 0.1
    LR_MILESTONE = 0.3          # fraction of stage epochs at which the x0.1 fires
    WARMUP_EPOCHS = 5
    MOMENTUM = 0.937
    WEIGHT_DECAY = 5e-4
    EPOCHS_SPTBACKBONE = 100
    EPOCHS_DAM = 100
    EVAL_EVERY = 20

    train_path = "../../datasets/DAUB/train_DAUB.txt"
    val_path   = "../../datasets/DAUB/val_DAUB.txt"

In [ ]:
import os
import cv2
import torch
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

class MOCIDDataset(Dataset):
    def __init__(self, annotations_file, T=5, img_size=(512, 512), is_train=True):
        self.T = T
        self.img_size = img_size
        self.is_train = is_train
        self.clips = []

        sequences = defaultdict(list)
        if not os.path.exists(annotations_file):
            print(f"Warning: {annotations_file} not found.")
            return

        with open(annotations_file, 'r') as f:
            for line in f:
                parts = line.strip().split(' ')
                if len(parts) != 2:
                    continue
                img_path, bbox_str = parts[0], parts[1]
                seq_id = os.path.basename(os.path.dirname(img_path))
                frame_num = int(os.path.basename(img_path).split('.')[0])
                box_data = list(map(int, bbox_str.split(',')))
                sequences[seq_id].append({
                    'path': img_path, 'frame_num': frame_num,
                    'bbox': box_data[:4], 'class_id': box_data[4]
                })

        for seq_id, frames in sequences.items():
            frames = sorted(frames, key=lambda x: x['frame_num'])
            for i in range(len(frames)):
                idxs = [max(i - (self.T - 1) + j, 0) for j in range(self.T)]
                self.clips.append([frames[k] for k in idxs])

        print(f"Loaded {len(self.clips)} clips of length {self.T} from {annotations_file}")

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_data = self.clips[idx]
        imgs = []

        target_info = clip_data[-1]
        orig_xmin, orig_ymin, orig_xmax, orig_ymax = target_info['bbox']
        target_class = target_info['class_id']

        sample_img = cv2.imread(clip_data[0]['path'], cv2.IMREAD_COLOR)
        if sample_img is None:
            raise FileNotFoundError(f"Could not read image at {clip_data[0]['path']}")

        orig_h, orig_w = sample_img.shape[:2]
        scale_x = self.img_size[0] / orig_w
        scale_y = self.img_size[1] / orig_h

        xmin = int(orig_xmin * scale_x); ymin = int(orig_ymin * scale_y)
        xmax = int(orig_xmax * scale_x); ymax = int(orig_ymax * scale_y)

        do_flip = self.is_train and (random.random() > 0.5)
        if do_flip:
            xmin, xmax = self.img_size[0] - xmax, self.img_size[0] - xmin

        for frame in clip_data:
            img = cv2.imread(frame['path'], cv2.IMREAD_COLOR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, self.img_size)
            if do_flip:
                img = cv2.flip(img, 1)
            img = img.astype(np.float32) / 255.0
            img = np.transpose(img, (2, 0, 1))
            imgs.append(img)

        imgs_tensor = torch.tensor(np.array(imgs), dtype=torch.float32)
        target = {
            'boxes': torch.tensor([[xmin, ymin, xmax, ymax]], dtype=torch.float32),
            'labels': torch.tensor([target_class], dtype=torch.int64)
        }
        return imgs_tensor, target


def collate_mocid(batch):
    clips = torch.stack([b[0] for b in batch])
    labels = []
    for _, tgt in batch:
        bx = tgt['boxes']
        cx, cy = (bx[:, 0] + bx[:, 2]) / 2, (bx[:, 1] + bx[:, 3]) / 2
        w,  h  = bx[:, 2] - bx[:, 0], bx[:, 3] - bx[:, 1]
        cls = tgt['labels'].float()
        labels.append(torch.stack([cx, cy, w, h, cls], dim=1))
    return clips, labels

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiLU(nn.Module):
    @staticmethod
    def forward(x):
        return x * torch.sigmoid(x)


def get_activation(name="silu", inplace=True):
    if name == "silu":
        module = SiLU()
    elif name == "relu":
        module = nn.ReLU(inplace=inplace)
    elif name == "lrelu":
        module = nn.LeakyReLU(0.1, inplace=inplace)
    elif name == "sigmoid":
        module = nn.Sigmoid()
    else:
        raise AttributeError("Unsupported act type: {}".format(name))
    return module


# CSPnet convolution block
class BaseConv(nn.Module):
    def __init__(self, in_channels, out_channels, ksize, stride, groups=1, bias=False, act="silu"):
        super().__init__()
        pad         = (ksize - 1) // 2
        self.conv   = nn.Conv2d(in_channels, out_channels, kernel_size=ksize, stride=stride, padding=pad, groups=groups, bias=bias)
        self.bn     = nn.BatchNorm2d(out_channels, eps=0.001, momentum=0.03)
        self.act    = get_activation(act, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def fuseforward(self, x):
        return self.act(self.conv(x))

class Bottleneck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        hidden_channels = out_channels // 2
        self.conv1 = BaseConv(in_channels, hidden_channels, 1, stride=1)
        self.conv2 = BaseConv(hidden_channels, out_channels, 3, stride=1)

    def forward(self, x):
        return x + self.conv2(self.conv1(x))

class CSPLayer(nn.Module):
    def __init__(self, in_channels, out_channels, num_bottlenecks=1):
        super().__init__()
        hidden_channels = out_channels // 2
        
        self.conv1 = BaseConv(in_channels, hidden_channels, 1, stride=1)
        self.conv2 = BaseConv(in_channels, hidden_channels, 1, stride=1)
        
        
        self.bottlenecks = nn.Sequential(
            *[Bottleneck(hidden_channels, hidden_channels) for _ in range(num_bottlenecks)]
        )

        self.conv3 = BaseConv(2 * hidden_channels, out_channels, 1, stride=1)

    def forward(self, x):
        # main path
        x_1 = self.conv1(x)
        x_1 = self.bottlenecks(x_1)

        # bypass path
        x_2 = self.conv2(x)

        # concatenate
        out = torch.cat((x_1, x_2), dim=1)
        return self.conv3(out)


In [ ]:

# ==========================================
# 2. FISTA Block
# ==========================================


class FISTABlock(nn.Module):
    def __init__(self, channels, num_frames, height, width, ksize=3):
        super().__init__()
        self.C, self.T, self.k = channels, num_frames, ksize
        Wf = width // 2 + 1

        # K  (Eq.1): complex spatial global filter, shape (C,H,Wf)
        self.spatial_filter  = nn.Parameter(torch.randn(channels, height, Wf, 2) * 0.02)
        # Kt (Eq.3): complex temporal global filter, shape (T,C,1,1)
        self.temporal_filter = nn.Parameter(torch.randn(num_frames, channels, 1, 1, 2) * 0.02)

        # W_b (Eq.5): base kernel, shared across frames. (C_out, C_in, k, k), no bias.
        self.base_weight = nn.Parameter(torch.empty(channels, channels, ksize, ksize))
        nn.init.kaiming_uniform_(self.base_weight, a=5 ** 0.5)
        self.pad = (ksize - 1) // 2

        # FC (Eq.6): operates across the temporal dim, shared across channels.
        self.temporal_fc = nn.Linear(num_frames, num_frames)
        nn.init.zeros_(self.temporal_fc.weight)   # init alpha_t ~= 1 -> f_out ~= W_b * f
        nn.init.ones_(self.temporal_fc.bias)

    def _motion_context(self, f):                 # f:(B,T,C,H,W) -> M:(B,T,C,H,W)
        B, T, C, H, W = f.shape
        x = f.reshape(B * T, C, H, W).float()     # FFT in fp32

        # Eq.1  spatial DFT -> filter -> IDFT  (per frame)
        Fs = torch.fft.rfft2(x, dim=(-2, -1), norm='ortho')
        Fs = Fs * torch.view_as_complex(self.spatial_filter)
        fs = torch.fft.irfft2(Fs, s=(H, W), dim=(-2, -1), norm='ortho').reshape(B, T, C, H, W)

        # Eq.2-3  temporal DFT -> filter -> IDFT  (per pixel, over T)
        Ft = torch.fft.fft(fs, dim=1, norm='ortho')
        Ft = Ft * torch.view_as_complex(self.temporal_filter)
        f_hat = torch.fft.ifft(Ft, dim=1, norm='ortho').real

        # Eq.4  L2-normalize f_hat (channel-wise), gate original f
        f_hat = F.normalize(f_hat, p=2, dim=2)
        return f * f_hat.to(f.dtype)

    def _calibration(self, M):                    # Eq.6  M -> alpha_t (B,T,C,1,1)
        B, T, C, H, W = M.shape
        d = M.mean(dim=(-2, -1))                  # GAP_s -> (B,T,C)
        a = self.temporal_fc(d.permute(0, 2, 1))  # mix across T -> (B,C,T)
        return a.permute(0, 2, 1).reshape(B, T, C, 1, 1)

    def forward(self, f):
        B, T, C, H, W = f.shape
        k = self.k
        alpha = self._calibration(self._motion_context(f))            # (B,T,C,1,1)

        # Eq.5: W_t = alpha_t · W_b  (alpha on C_out), then f_out = W_t * f
        # W_t: (B,T,Cout,Cin,k,k)
        W_t = alpha.reshape(B, T, C, 1, 1, 1) * self.base_weight.reshape(1, 1, C, C, k, k)
        W_t = W_t.reshape(B * T * C, C, k, k)                         # (Cout_total, Cin/group, k, k)

        f_flat = f.reshape(1, B * T * C, H, W)                        # (1, Cin_total, H, W)
        out = F.conv2d(f_flat, W_t, padding=self.pad, groups=B * T)   # one kernel per (B,T)
        return out.reshape(B, T, C, H, W)


class FISTALayer(nn.Module):
    """
    Faithful implementation of the MOCID FISTA Layer (Figure 2, bottom-right).
    Input -> 3x3 Conv (Downsample)
          -> Branch 1: 1x1 Conv -----------------------> (+) -> Output
          -> Branch 2: 1x1 Conv -> FISTA block --------^
    """
    def __init__(self, in_channels, out_channels, num_frames, out_h, out_w):
        super().__init__()
        # 1. Spatial Downsampling (operates on folded B*T)
        self.downsample = BaseConv(in_channels, out_channels, ksize=3, stride=2)
        
        # 2. Top Branch (Purely Spatial)
        self.branch1 = BaseConv(out_channels, out_channels, ksize=1, stride=1)
        
        # 3. Bottom Branch (Spatio-Temporal)
        self.branch2_conv = BaseConv(out_channels, out_channels, ksize=1, stride=1)
        self.fista = FISTABlock(out_channels, num_frames, out_h, out_w)

    def forward(self, x):
        B, T, C, H, W = x.shape
        
        # Fold batch and time to process 2D spatial convolutions
        x_flat = x.reshape(B * T, C, H, W)
        
        # Initial 3x3 Downsample
        x_down = self.downsample(x_flat)
        _, Co, Ho, Wo = x_down.shape
        
        # --- TOP BRANCH ---
        out1 = self.branch1(x_down)
        # Reshape to 5D for fusion later
        out1_5d = out1.reshape(B, T, Co, Ho, Wo)

        # --- BOTTOM BRANCH ---
        out2_conv = self.branch2_conv(x_down)
        # Unfold back to 5D because FISTA requires the Temporal (T) dimension
        out2_fista_in = out2_conv.reshape(B, T, Co, Ho, Wo)
        out2_fista_out = self.fista(out2_fista_in)
        
        # --- FUSION (+) ---
        # Element-wise addition of the spatial branch and the motion-context branch
        out = out1_5d + out2_fista_out
        
        return out


base channels are not defined in paper, so used 64 (resnet uses 64)

In [ ]:

# ==========================================
# 3. Main Spatio-Temporal Backbone
# ==========================================
class SpatioTemporalBackbone(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, num_frames=5, img_size=512):
        super().__init__()
        self.stem  = BaseConv(in_channels, base_channels, 3, stride=2)          # /2
        self.dark2 = nn.Sequential(
            BaseConv(base_channels, base_channels * 2, 3, stride=2),            # /4
            CSPLayer(base_channels * 2, base_channels * 2, num_bottlenecks=2))

        s = img_size                                                            # resolutions
        h1, h2, h3 = s // 8, s // 16, s // 32                                   # 64, 32, 16
        self.fista_layer1 = FISTALayer(base_channels * 2, base_channels * 4, num_frames, h1, h1)
        self.fista_layer2 = FISTALayer(base_channels * 4, base_channels * 8, num_frames, h2, h2)
        self.fista_layer3 = FISTALayer(base_channels * 8, base_channels * 16, num_frames, h3, h3)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)              
        x = self.dark2(self.stem(x))
        _, C2, H2, W2 = x.shape
        x = x.reshape(B, T, C2, H2, W2)            

        f1 = self.fista_layer1(x)
        f2 = self.fista_layer2(f1)
        f3 = self.fista_layer3(f2)
        return f1, f2, f3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CDC3d(nn.Module):
    """3D Central Difference Conv (Yu et al. 2021), single-pass weight reparam.
       Non-cubic kernel supported; difference center given explicitly."""
    def __init__(self, in_ch, out_ch, kernel_size=(2, 3, 3), stride=1,
                 padding=(0, 1, 1), bias=False, theta=0.7, center=(1, 1, 1)):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size, stride, padding, bias=bias)
        self.theta, self.center = theta, center

    def forward(self, x):
        w = self.conv.weight
        w_sum = w.sum(dim=(2, 3, 4), keepdim=True)
        mask = torch.zeros_like(w)
        ct, ch, cw = self.center
        mask[:, :, ct, ch, cw] = 1.0
        cdc_w = w - self.theta * w_sum * mask
        return F.conv3d(x, cdc_w, self.conv.bias, self.conv.stride, self.conv.padding)

import math

class SDS(nn.Module):
    """Pure-Mamba selection with 3DCDC as the generator (paper's phi).
       3DCDC emits [dt_rank | B(N) | C(N)]; dt_proj expands Delta to d_inner.
       NOTE: low-rank Delta is a deliberate deviation from the paper's
       'Delta = 3DCDC(x)' (full-width), adopted for Mamba-fidelity + param budget."""
    def __init__(self, d_inner, d_state=16, theta=0.7, dt_rank=None,
                 dt_min=1e-3, dt_max=1e-1):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.dt_rank = dt_rank or math.ceil(d_inner / 16)

        self.cdc3d = CDC3d(d_inner, self.dt_rank + 2 * d_state,
                           kernel_size=(2, 3, 3), padding=(0, 1, 1),
                           theta=theta, center=(1, 1, 1))          # kt=2, depth-1 native

        # dt_proj as 1x1 conv (== per-position Linear), stock Mamba init
        self.dt_proj = nn.Conv2d(self.dt_rank, d_inner, 1, bias=True)
        std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -std, std)
        dt = torch.exp(torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min))
                       + math.log(dt_min)).clamp(min=1e-4)
        with torch.no_grad():
            self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))  # inverse softplus

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))

    @property
    def A(self): return -torch.exp(self.A_log.float())              # (d_inner, N)

    def forward(self, F_T, F_R):
        out = self.cdc3d(torch.stack([F_R, F_T], dim=2)).squeeze(2)  # (B, dt_rank+2N, H, W)
        dt, Bp, Cp = torch.split(out, [self.dt_rank, self.d_state, self.d_state], dim=1)
        delta = self.dt_proj(dt)                                     # (B, d_inner, H, W) PRE-softplus
        return (delta, Bp, Cp)                                       # TIS applies softplus in the scan


try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn, selective_scan_ref
except ImportError:
    selective_scan_fn = None

    def selective_scan_ref(u, delta, A, B, C, delta_softplus=True, **kw):
        if delta_softplus: delta = F.softplus(delta)
        b, d, L = u.shape
        dA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A))
        dB = torch.einsum('bdl,bnl->bdln', delta, B)
        h = u.new_zeros(b, d, A.shape[1]); ys = []
        for i in range(L):
            h = dA[:, :, i] * h + dB[:, :, i] * u[:, :, i:i+1]
            ys.append(torch.einsum('bdn,bn->bd', h, C[:, :, i]))
        return torch.stack(ys, -1)

def _ssm(*a, **k):                       # pick kernel on CUDA, ref otherwise
    fn = selective_scan_fn if (selective_scan_fn and a[0].is_cuda) else selective_scan_ref
    return fn(*a, **k)


class TIS(nn.Module):
    """Interleave features + raw per-frame params, then bidirectional selective scan via kernel."""
    @staticmethod
    def _pool(p, k):
        *lead, H, W = p.shape
        x = F.avg_pool2d(p.reshape(-1, 1, H, W), k)
        return x.reshape(*lead, x.shape[-2], x.shape[-1])

    @staticmethod
    def _il(r, t): return torch.stack([r, t], dim=-1).reshape(*r.shape[:-1], -1)

    def _scan(self, u, delta, A, B, C):
        yf = _ssm(u, delta, A, B, C, delta_softplus=True)
        yb = _ssm(u.flip(-1), delta.flip(-1), A, B.flip(-1), C.flip(-1), delta_softplus=True).flip(-1)
        return yf + yb

    def _branch(self, xT, xR, p, A, k):                # p = (delta,B,C), one shared set
        d_, B_, C_ = p
        Bb, d, H, W = xT.shape
        u  = self._il(self._pool(xR, k).flatten(2), self._pool(xT, k).flatten(2))     # (B,d,L)
        sh = lambda t: self._il(self._pool(t, k).flatten(-2), self._pool(t, k).flatten(-2))
        y  = self._scan(u, sh(d_), A, sh(B_), sh(C_))                                  # (B,d,L)
        return y.reshape(Bb, d, H, W)

    def forward(self, xT, xR, p, A):
        return self._branch(xT, xR, p, A, (1, 2)) + self._branch(xT, xR, p, A, (2, 1))

In [ ]:

class DAM(nn.Module):
    """ in_proj(d->2m) -> [x: dwconv+SiLU -> TIDS] * SiLU(z) -> out_proj(m->d)."""
    def __init__(self, d_model, d_state=16, expand=1, d_conv=3, theta=0.7):
        super().__init__()
        m = int(expand * d_model)
        self.d_inner = m
        self.norm     = nn.LayerNorm(d_model)
        self.in_proj  = nn.Linear(d_model, 2 * m, bias=False)        # x and z in one matmul
        self.dwconv   = nn.Conv2d(m, m, d_conv, padding=d_conv // 2, groups=m)
        self.sds      = SDS(m, d_state, theta)
        self.tis      = TIS()
        self.out_proj = nn.Linear(m, d_model, bias=False)

    def _proj(self, feat):                                           # LN -> in_proj -> split x,z
        h = self.norm(feat.permute(0, 2, 3, 1))
        x, z = self.in_proj(h).chunk(2, dim=-1)
        return x.permute(0, 3, 1, 2), z.permute(0, 3, 1, 2)

    def forward(self, feat_t, feat_r):
        xT, zT = self._proj(feat_t)
        xR, _  = self._proj(feat_r)                                  # gate comes from target only
        xT = F.silu(self.dwconv(xT)); xR = F.silu(self.dwconv(xR))
        p = self.sds(xT, xR)
        y = self.tis(xT, xR, p, self.sds.A) * F.silu(zT)
        return self.out_proj(y.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

In [ ]:
class DisplacementNet(nn.Module):
    def __init__(self, channels, d_state=16):
        super().__init__()
        self.dams = nn.ModuleList([DAM(c, d_state) for c in channels])

    def forward(self, feats):                              # feats: 3x (B,T,C,H,W)
        f_T, f_f = [], []
        for s, f in enumerate(feats):
            F_T = f[:, -1]                                 # target (last frame)
            
            # FIX: Changed .mean(0) to .max(0)[0] for Max Temporal Pooling
            F_f = torch.stack(
                [self.dams[s](F_T, f[:, i]) for i in range(f.shape[1] - 1)], 0
            ).max(0)[0]                                    # temporal pooling over T-1 DAMs
            
            f_T.append(F_T); f_f.append(F_f)
        return f_T, f_f                                    # clip-level, frame-level

In [ ]:
class YOLOXHead(nn.Module):
    def __init__(self, num_classes, width = 1.0, in_channels = [16, 32, 64], act = "silu"):
        super().__init__()
        Conv            =  BaseConv
        
        self.cls_convs  = nn.ModuleList()
        self.reg_convs  = nn.ModuleList()
        self.cls_preds  = nn.ModuleList()
        self.reg_preds  = nn.ModuleList()
        self.obj_preds  = nn.ModuleList()
        self.stems      = nn.ModuleList()

        for i in range(len(in_channels)):
            self.stems.append(BaseConv(in_channels = int(in_channels[i] * width), out_channels = int(256 * width), ksize = 1, stride = 1, act = act))
            self.cls_convs.append(nn.Sequential(*[
                Conv(in_channels = int(256 * width), out_channels = int(256 * width), ksize = 3, stride = 1, act = act), 
                Conv(in_channels = int(256 * width), out_channels = int(256 * width), ksize = 3, stride = 1, act = act), 
            ]))
            self.cls_preds.append(
                nn.Conv2d(in_channels = int(256 * width), out_channels = num_classes, kernel_size = 1, stride = 1, padding = 0)
            )
            

            self.reg_convs.append(nn.Sequential(*[
                Conv(in_channels = int(256 * width), out_channels = int(256 * width), ksize = 3, stride = 1, act = act), 
                Conv(in_channels = int(256 * width), out_channels = int(256 * width), ksize = 3, stride = 1, act = act)
            ]))
            self.reg_preds.append(
                nn.Conv2d(in_channels = int(256 * width), out_channels = 4, kernel_size = 1, stride = 1, padding = 0)
            )
            self.obj_preds.append(
                nn.Conv2d(in_channels = int(256 * width), out_channels = 1, kernel_size = 1, stride = 1, padding = 0)
            )

    def forward(self, inputs):
        #---------------------------------------------------#
        #   inputs输入
        #   P3_out  80, 80, 256
        #   P4_out  40, 40, 512
        #   P5_out  20, 20, 1024
        #---------------------------------------------------#
        outputs = []
        for k, x in enumerate(inputs):
            #---------------------------------------------------#
            #   利用1x1卷积进行通道整合
            #---------------------------------------------------#
            x       = self.stems[k](x)
            #---------------------------------------------------#
            #   利用两个卷积标准化激活函数来进行特征提取
            #---------------------------------------------------#
            cls_feat    = self.cls_convs[k](x)
            #---------------------------------------------------#
            #   判断特征点所属的种类
            #   80, 80, num_classes
            #   40, 40, num_classes
            #   20, 20, num_classes
            #---------------------------------------------------#
            cls_output  = self.cls_preds[k](cls_feat)

            #---------------------------------------------------#
            #   利用两个卷积标准化激活函数来进行特征提取
            #---------------------------------------------------#
            reg_feat    = self.reg_convs[k](x)
            #---------------------------------------------------#
            #   特征点的回归系数
            #   reg_pred 80, 80, 4
            #   reg_pred 40, 40, 4
            #   reg_pred 20, 20, 4
            #---------------------------------------------------#
            reg_output  = self.reg_preds[k](reg_feat)
            #---------------------------------------------------#
            #   判断特征点是否有对应的物体
            #   obj_pred 80, 80, 1
            #   obj_pred 40, 40, 1
            #   obj_pred 20, 20, 1
            #---------------------------------------------------#
            obj_output  = self.obj_preds[k](reg_feat)

            output      = torch.cat([reg_output, obj_output, cls_output], 1)
            outputs.append(output)
        return outputs

In [ ]:
class YOLOLoss(nn.Module):    
    def __init__(self, num_classes, fp16, strides=[8, 16, 32]):
        super().__init__()
        self.num_classes        = num_classes
        self.strides            = strides

        self.bcewithlog_loss    = nn.BCEWithLogitsLoss(reduction="none")
        self.iou_loss           = IOUloss(reduction="none")
        self.grids              = [torch.zeros(1)] * len(strides)
        self.fp16               = fp16

    def forward(self, inputs, labels=None):
        outputs             = []
        x_shifts            = []
        y_shifts            = []
        expanded_strides    = []

        #-----------------------------------------------#
        # inputs    [[batch_size, num_classes + 5, 20, 20]
        #            [batch_size, num_classes + 5, 40, 40]
        #            [batch_size, num_classes + 5, 80, 80]]
        # outputs   [[batch_size, 400, num_classes + 5]
        #            [batch_size, 1600, num_classes + 5]
        #            [batch_size, 6400, num_classes + 5]]
        # x_shifts  [[batch_size, 400]
        #            [batch_size, 1600]
        #            [batch_size, 6400]]
        #-----------------------------------------------#
        for k, (stride, output) in enumerate(zip(self.strides, inputs)):
            output, grid = self.get_output_and_grid(output, k, stride)
            x_shifts.append(grid[:, :, 0])
            y_shifts.append(grid[:, :, 1])
            expanded_strides.append(torch.ones_like(grid[:, :, 0]) * stride)
            outputs.append(output)

        return self.get_losses(x_shifts, y_shifts, expanded_strides, labels, torch.cat(outputs, 1))

    def get_output_and_grid(self, output, k, stride):
        grid            = self.grids[k]
        hsize, wsize    = output.shape[-2:]
        if grid.shape[2:4] != output.shape[2:4]:
            yv, xv          = torch.meshgrid([torch.arange(hsize), torch.arange(wsize)], indexing='ij')
            grid            = torch.stack((xv, yv), 2).view(1, hsize, wsize, 2).type(output.type())
            self.grids[k]   = grid
        grid                = grid.view(1, -1, 2)

        output              = output.flatten(start_dim=2).permute(0, 2, 1)
        
        # -----------------------------------------------------------
        # ROBUST FIX: Build a new tensor from pieces using torch.cat
        # -----------------------------------------------------------
        xy = (output[..., :2] + grid.type_as(output)) * stride
        wh = torch.exp(torch.clamp(output[..., 2:4], max=20.0)) * stride
        rest = output[..., 4:]
        
        # This creates a completely new tensor in memory, satisfying autograd
        output = torch.cat([xy, wh, rest], dim=-1)
        
        return output, grid

    def get_losses(self, x_shifts, y_shifts, expanded_strides, labels, outputs):
        #-----------------------------------------------#
        #   [batch, n_anchors_all, 4]
        #-----------------------------------------------#
        bbox_preds  = outputs[:, :, :4]  
        #-----------------------------------------------#
        #   [batch, n_anchors_all, 1]
        #-----------------------------------------------#
        obj_preds   = outputs[:, :, 4:5]
        #-----------------------------------------------#
        #   [batch, n_anchors_all, n_cls]
        #-----------------------------------------------#
        cls_preds   = outputs[:, :, 5:]  

        total_num_anchors   = outputs.shape[1]
        #-----------------------------------------------#
        #   x_shifts            [1, n_anchors_all]
        #   y_shifts            [1, n_anchors_all]
        #   expanded_strides    [1, n_anchors_all]
        #-----------------------------------------------#
        x_shifts            = torch.cat(x_shifts, 1).type_as(outputs)
        y_shifts            = torch.cat(y_shifts, 1).type_as(outputs)
        expanded_strides    = torch.cat(expanded_strides, 1).type_as(outputs)

        cls_targets = []
        reg_targets = []
        obj_targets = []
        fg_masks    = []

        num_fg  = 0.0
        for batch_idx in range(outputs.shape[0]):
            num_gt          = len(labels[batch_idx])
            if num_gt == 0:
                cls_target  = outputs.new_zeros((0, self.num_classes))
                reg_target  = outputs.new_zeros((0, 4))
                obj_target  = outputs.new_zeros((total_num_anchors, 1))
                fg_mask     = outputs.new_zeros(total_num_anchors).bool()
            else:
                #-----------------------------------------------#
                #   gt_bboxes_per_image     [num_gt, num_classes]
                #   gt_classes              [num_gt]
                #   bboxes_preds_per_image  [n_anchors_all, 4]
                #   cls_preds_per_image     [n_anchors_all, num_classes]
                #   obj_preds_per_image     [n_anchors_all, 1]
                #-----------------------------------------------#
                gt_bboxes_per_image     = labels[batch_idx][..., :4].type_as(outputs)
                gt_classes              = labels[batch_idx][..., 4].type_as(outputs)
                bboxes_preds_per_image  = bbox_preds[batch_idx]
                cls_preds_per_image     = cls_preds[batch_idx]
                obj_preds_per_image     = obj_preds[batch_idx]

                gt_matched_classes, fg_mask, pred_ious_this_matching, matched_gt_inds, num_fg_img = self.get_assignments( 
                    num_gt, total_num_anchors, gt_bboxes_per_image, gt_classes, bboxes_preds_per_image, cls_preds_per_image, obj_preds_per_image,
                    expanded_strides, x_shifts, y_shifts, 
                )
                # torch.cuda.empty_cache()
                num_fg      += num_fg_img
                cls_target  = F.one_hot(gt_matched_classes.to(torch.int64), self.num_classes).float() * pred_ious_this_matching.unsqueeze(-1)
                obj_target  = fg_mask.unsqueeze(-1)
                reg_target  = gt_bboxes_per_image[matched_gt_inds]
            cls_targets.append(cls_target)
            reg_targets.append(reg_target)
            obj_targets.append(obj_target.type(cls_target.type()))
            fg_masks.append(fg_mask)

        cls_targets = torch.cat(cls_targets, 0)
        reg_targets = torch.cat(reg_targets, 0)
        obj_targets = torch.cat(obj_targets, 0)
        fg_masks    = torch.cat(fg_masks, 0)

        num_fg      = max(num_fg, 1)
        loss_iou    = (self.iou_loss(bbox_preds.view(-1, 4)[fg_masks], reg_targets)).sum()
        loss_obj    = (self.bcewithlog_loss(obj_preds.view(-1, 1), obj_targets)).sum()
        loss_cls    = (self.bcewithlog_loss(cls_preds.view(-1, self.num_classes)[fg_masks], cls_targets)).sum()
        # loss_obj    = (sigmoid_focal_loss(obj_preds.view(-1, 1), obj_targets)).sum()
        # loss_cls    = (sigmoid_focal_loss(cls_preds.view(-1, self.num_classes)[fg_masks], cls_targets)).sum()
        reg_weight  = 5.0
        loss = reg_weight * loss_iou + loss_obj + loss_cls

        return loss / num_fg

    @torch.no_grad()
    def get_assignments(self, num_gt, total_num_anchors, gt_bboxes_per_image, gt_classes, bboxes_preds_per_image, cls_preds_per_image, obj_preds_per_image, expanded_strides, x_shifts, y_shifts):
        #-------------------------------------------------------#
        #   fg_mask                 [n_anchors_all]
        #   is_in_boxes_and_center  [num_gt, len(fg_mask)]
        #-------------------------------------------------------#
        fg_mask, is_in_boxes_and_center = self.get_in_boxes_info(gt_bboxes_per_image, expanded_strides, x_shifts, y_shifts, total_num_anchors, num_gt)

        #-------------------------------------------------------#
        #   fg_mask                 [n_anchors_all]
        #   bboxes_preds_per_image  [fg_mask, 4]
        #   cls_preds_              [fg_mask, num_classes]
        #   obj_preds_              [fg_mask, 1]
        #-------------------------------------------------------#
        bboxes_preds_per_image  = bboxes_preds_per_image[fg_mask]
        cls_preds_              = cls_preds_per_image[fg_mask]
        obj_preds_              = obj_preds_per_image[fg_mask]
        num_in_boxes_anchor     = bboxes_preds_per_image.shape[0]


        # ======================================================= #
        # ADD THIS SAFETY CHECK TO PREVENT CUDA ASSERT ERRORS
        # ======================================================= #
        if num_in_boxes_anchor == 0:
            return (
                gt_classes.new_zeros((0,), dtype=torch.long),
                gt_bboxes_per_image.new_zeros((total_num_anchors,), dtype=torch.bool),
                gt_bboxes_per_image.new_zeros((0,), dtype=torch.float),
                gt_classes.new_zeros((0,), dtype=torch.long),
                0
            )

        #-------------------------------------------------------#
        #   pair_wise_ious      [num_gt, fg_mask]
        #-------------------------------------------------------#
        pair_wise_ious      = self.bboxes_iou(gt_bboxes_per_image, bboxes_preds_per_image, False)
        pair_wise_ious_loss = -torch.log(pair_wise_ious + 1e-8)
        
        #-------------------------------------------------------#
        #   cls_preds_          [num_gt, fg_mask, num_classes]
        #   gt_cls_per_image    [num_gt, fg_mask, num_classes]
        #-------------------------------------------------------#
        if self.fp16:
            with torch.cuda.amp.autocast(enabled=False):
                # FIX: Removed .sigmoid_() and replaced with .sigmoid()
                cls_preds_          = cls_preds_.float().unsqueeze(0).repeat(num_gt, 1, 1).sigmoid() * obj_preds_.unsqueeze(0).repeat(num_gt, 1, 1).sigmoid()
                gt_cls_per_image    = F.one_hot(gt_classes.to(torch.int64), self.num_classes).float().unsqueeze(1).repeat(1, num_in_boxes_anchor, 1)
                pair_wise_cls_loss  = F.binary_cross_entropy(cls_preds_.sqrt_(), gt_cls_per_image, reduction="none").sum(-1)
        else:
            # FIX: Removed .sigmoid_() and replaced with .sigmoid()
            cls_preds_          = cls_preds_.float().unsqueeze(0).repeat(num_gt, 1, 1).sigmoid() * obj_preds_.unsqueeze(0).repeat(num_gt, 1, 1).sigmoid()
            gt_cls_per_image    = F.one_hot(gt_classes.to(torch.int64), self.num_classes).float().unsqueeze(1).repeat(1, num_in_boxes_anchor, 1)
            pair_wise_cls_loss  = F.binary_cross_entropy(cls_preds_.sqrt_(), gt_cls_per_image, reduction="none").sum(-1)
            del cls_preds_

        cost = pair_wise_cls_loss + 3.0 * pair_wise_ious_loss + 100000.0 * (~is_in_boxes_and_center).float()

        num_fg, gt_matched_classes, pred_ious_this_matching, matched_gt_inds = self.dynamic_k_matching(cost, pair_wise_ious, gt_classes, num_gt, fg_mask)
        del pair_wise_cls_loss, cost, pair_wise_ious, pair_wise_ious_loss
        return gt_matched_classes, fg_mask, pred_ious_this_matching, matched_gt_inds, num_fg
    
    def bboxes_iou(self, bboxes_a, bboxes_b, xyxy=True):
        if bboxes_a.shape[1] != 4 or bboxes_b.shape[1] != 4:
            raise IndexError

        if xyxy:
            tl = torch.max(bboxes_a[:, None, :2], bboxes_b[:, :2])
            br = torch.min(bboxes_a[:, None, 2:], bboxes_b[:, 2:])
            area_a = torch.prod(bboxes_a[:, 2:] - bboxes_a[:, :2], 1)
            area_b = torch.prod(bboxes_b[:, 2:] - bboxes_b[:, :2], 1)
        else:
            tl = torch.max(
                (bboxes_a[:, None, :2] - bboxes_a[:, None, 2:] / 2),
                (bboxes_b[:, :2] - bboxes_b[:, 2:] / 2),
            )
            br = torch.min(
                (bboxes_a[:, None, :2] + bboxes_a[:, None, 2:] / 2),
                (bboxes_b[:, :2] + bboxes_b[:, 2:] / 2),
            )

            area_a = torch.prod(bboxes_a[:, 2:], 1)
            area_b = torch.prod(bboxes_b[:, 2:], 1)
        en = (tl < br).type(tl.type()).prod(dim=2)
        area_i = torch.prod(br - tl, 2) * en
        return area_i / (area_a[:, None] + area_b - area_i)

    def get_in_boxes_info(self, gt_bboxes_per_image, expanded_strides, x_shifts, y_shifts, total_num_anchors, num_gt, center_radius = 2.5):
        #-------------------------------------------------------#
        #   expanded_strides_per_image  [n_anchors_all]
        #   x_centers_per_image         [num_gt, n_anchors_all]
        #   x_centers_per_image         [num_gt, n_anchors_all]
        #-------------------------------------------------------#
        expanded_strides_per_image  = expanded_strides[0]
        x_centers_per_image         = ((x_shifts[0] + 0.5) * expanded_strides_per_image).unsqueeze(0).repeat(num_gt, 1)
        y_centers_per_image         = ((y_shifts[0] + 0.5) * expanded_strides_per_image).unsqueeze(0).repeat(num_gt, 1)

        #-------------------------------------------------------#
        #   gt_bboxes_per_image_x       [num_gt, n_anchors_all]
        #-------------------------------------------------------#
        gt_bboxes_per_image_l = (gt_bboxes_per_image[:, 0] - 0.5 * gt_bboxes_per_image[:, 2]).unsqueeze(1).repeat(1, total_num_anchors)
        gt_bboxes_per_image_r = (gt_bboxes_per_image[:, 0] + 0.5 * gt_bboxes_per_image[:, 2]).unsqueeze(1).repeat(1, total_num_anchors)
        gt_bboxes_per_image_t = (gt_bboxes_per_image[:, 1] - 0.5 * gt_bboxes_per_image[:, 3]).unsqueeze(1).repeat(1, total_num_anchors)
        gt_bboxes_per_image_b = (gt_bboxes_per_image[:, 1] + 0.5 * gt_bboxes_per_image[:, 3]).unsqueeze(1).repeat(1, total_num_anchors)

        #-------------------------------------------------------#
        #   bbox_deltas     [num_gt, n_anchors_all, 4]
        #-------------------------------------------------------#
        b_l = x_centers_per_image - gt_bboxes_per_image_l
        b_r = gt_bboxes_per_image_r - x_centers_per_image
        b_t = y_centers_per_image - gt_bboxes_per_image_t
        b_b = gt_bboxes_per_image_b - y_centers_per_image
        bbox_deltas = torch.stack([b_l, b_t, b_r, b_b], 2)

        #-------------------------------------------------------#
        #   is_in_boxes     [num_gt, n_anchors_all]
        #   is_in_boxes_all [n_anchors_all]
        #-------------------------------------------------------#
        is_in_boxes     = bbox_deltas.min(dim=-1).values > 0.0
        is_in_boxes_all = is_in_boxes.sum(dim=0) > 0

        gt_bboxes_per_image_l = (gt_bboxes_per_image[:, 0]).unsqueeze(1).repeat(1, total_num_anchors) - center_radius * expanded_strides_per_image.unsqueeze(0)
        gt_bboxes_per_image_r = (gt_bboxes_per_image[:, 0]).unsqueeze(1).repeat(1, total_num_anchors) + center_radius * expanded_strides_per_image.unsqueeze(0)
        gt_bboxes_per_image_t = (gt_bboxes_per_image[:, 1]).unsqueeze(1).repeat(1, total_num_anchors) - center_radius * expanded_strides_per_image.unsqueeze(0)
        gt_bboxes_per_image_b = (gt_bboxes_per_image[:, 1]).unsqueeze(1).repeat(1, total_num_anchors) + center_radius * expanded_strides_per_image.unsqueeze(0)

        #-------------------------------------------------------#
        #   center_deltas   [num_gt, n_anchors_all, 4]
        #-------------------------------------------------------#
        c_l = x_centers_per_image - gt_bboxes_per_image_l
        c_r = gt_bboxes_per_image_r - x_centers_per_image
        c_t = y_centers_per_image - gt_bboxes_per_image_t
        c_b = gt_bboxes_per_image_b - y_centers_per_image
        center_deltas       = torch.stack([c_l, c_t, c_r, c_b], 2)

        #-------------------------------------------------------#
        #   is_in_centers       [num_gt, n_anchors_all]
        #   is_in_centers_all   [n_anchors_all]
        #-------------------------------------------------------#
        is_in_centers       = center_deltas.min(dim=-1).values > 0.0
        is_in_centers_all   = is_in_centers.sum(dim=0) > 0

        #-------------------------------------------------------#
        #   is_in_boxes_anchor      [n_anchors_all]
        #   is_in_boxes_and_center  [num_gt, is_in_boxes_anchor]
        #-------------------------------------------------------#
        is_in_boxes_anchor      = is_in_boxes_all | is_in_centers_all
        is_in_boxes_and_center  = is_in_boxes[:, is_in_boxes_anchor] & is_in_centers[:, is_in_boxes_anchor]
        return is_in_boxes_anchor, is_in_boxes_and_center

    def dynamic_k_matching(self, cost, pair_wise_ious, gt_classes, num_gt, fg_mask):
        #-------------------------------------------------------#
        #   cost                [num_gt, fg_mask]
        #   pair_wise_ious      [num_gt, fg_mask]
        #   gt_classes          [num_gt]        
        #   fg_mask             [n_anchors_all]
        #   matching_matrix     [num_gt, fg_mask]
        #-------------------------------------------------------#
        matching_matrix         = torch.zeros_like(cost)

        #------------------------------------------------------------#
        #   选取iou最大的n_candidate_k个点
        #   然后求和，判断应该有多少点用于该框预测
        #   topk_ious           [num_gt, n_candidate_k]
        #   dynamic_ks          [num_gt]
        #   matching_matrix     [num_gt, fg_mask]
        #------------------------------------------------------------#
        n_candidate_k           = min(10, pair_wise_ious.size(1))
        topk_ious, _            = torch.topk(pair_wise_ious, n_candidate_k, dim=1)
        dynamic_ks              = torch.clamp(topk_ious.sum(1).int(), min=1, max=pair_wise_ious.size(1))
        
        for gt_idx in range(num_gt):
            #------------------------------------------------------------#
            #   给每个真实框选取最小的动态k个点
            #------------------------------------------------------------#
            _, pos_idx = torch.topk(cost[gt_idx], k=dynamic_ks[gt_idx].item(), largest=False)
            matching_matrix[gt_idx][pos_idx] = 1.0
        del topk_ious, dynamic_ks, pos_idx

        #------------------------------------------------------------#
        #   anchor_matching_gt  [fg_mask]
        #------------------------------------------------------------#
        anchor_matching_gt = matching_matrix.sum(0)
        if (anchor_matching_gt > 1).sum() > 0:
            #------------------------------------------------------------#
            #   当某一个特征点指向多个真实框的时候
            #   选取cost最小的真实框。
            #------------------------------------------------------------#
            _, cost_argmin = torch.min(cost[:, anchor_matching_gt > 1], dim=0)
            matching_matrix[:, anchor_matching_gt > 1] *= 0.0
            matching_matrix[cost_argmin, anchor_matching_gt > 1] = 1.0
        #------------------------------------------------------------#
        #   fg_mask_inboxes  [fg_mask]
        #   num_fg为正样本的特征点个数
        #------------------------------------------------------------#
        fg_mask_inboxes = matching_matrix.sum(0) > 0.0
        num_fg          = fg_mask_inboxes.sum().item()

        #------------------------------------------------------------#
        #   对fg_mask进行更新
        #------------------------------------------------------------#
        fg_mask[fg_mask.clone()] = fg_mask_inboxes

        #------------------------------------------------------------#
        #   获得特征点对应的物品种类
        #------------------------------------------------------------#
        matched_gt_inds     = matching_matrix[:, fg_mask_inboxes].argmax(0)
        gt_matched_classes  = gt_classes[matched_gt_inds]

        pred_ious_this_matching = (matching_matrix * pair_wise_ious).sum(0)[fg_mask_inboxes]
        return num_fg, gt_matched_classes, pred_ious_this_matching, matched_gt_inds



In [ ]:
class IOUloss(nn.Module):
    def __init__(self, reduction="none", loss_type="iou"):
        super(IOUloss, self).__init__()
        self.reduction = reduction
        self.loss_type = loss_type

    def forward(self, pred, target):
        assert pred.shape[0] == target.shape[0]

        pred = pred.view(-1, 4)
        target = target.view(-1, 4)
        tl = torch.max(
            (pred[:, :2] - pred[:, 2:] / 2), (target[:, :2] - target[:, 2:] / 2)
        )
        br = torch.min(
            (pred[:, :2] + pred[:, 2:] / 2), (target[:, :2] + target[:, 2:] / 2)
        )

        area_p = torch.prod(pred[:, 2:], 1)
        area_g = torch.prod(target[:, 2:], 1)

        en = (tl < br).type(tl.type()).prod(dim=1)
        area_i = torch.prod(br - tl, 1) * en
        area_u = area_p + area_g - area_i
        iou = (area_i) / (area_u + 1e-16)

        if self.loss_type == "iou":
            loss = 1 - iou ** 2
        elif self.loss_type == "giou":
            c_tl = torch.min(
                (pred[:, :2] - pred[:, 2:] / 2), (target[:, :2] - target[:, 2:] / 2)
            )
            c_br = torch.max(
                (pred[:, :2] + pred[:, 2:] / 2), (target[:, :2] + target[:, 2:] / 2)
            )
            area_c = torch.prod(c_br - c_tl, 1)
            giou = iou - (area_c - area_u) / area_c.clamp(1e-16)
            loss = 1 - giou.clamp(min=-1.0, max=1.0)
        elif self.loss_type == 'ciou':
            b1_cxy = pred[:,:2]
            b2_cxy = target[:,:2]
            # 计算中心的差距
            center_distance = torch.sum(torch.pow((b1_cxy - b2_cxy), 2), axis=-1)
            # 找到包裹两个框的最小框的左上角和右下角
            enclose_mins = torch.min((pred[:, :2] - pred[:, 2:] / 2), (target[:, :2] - target[:, 2:] / 2))
            enclose_maxes = torch.max((pred[:, :2] + pred[:, 2:] / 2), (target[:, :2] + target[:, 2:] / 2))
            enclose_wh = torch.max(enclose_maxes - enclose_mins, torch.zeros_like(br))
            # 计算对角线距离
            enclose_diagonal = torch.sum(torch.pow(enclose_wh,2), axis=-1)
            ciou = iou - 1.0 * (center_distance) / torch.clamp(enclose_diagonal,min = 1e-6)
            v = (4 / (torch.pi ** 2)) * torch.pow((torch.atan(pred[:, 2]/torch.clamp(pred[:, 3],min = 1e-6)) - torch.atan(target[:, 2]/torch.clamp(target[:, 3],min = 1e-6))), 2)
            alpha = v / torch.clamp((1.0 - iou + v),min=1e-6)
            ciou = ciou - alpha * v
            loss = 1 - ciou.clamp(min=-1.0, max=1.0)

        if self.reduction == "mean":
            loss = loss.mean()
        elif self.reduction == "sum":
            loss = loss.sum()

        return loss

In [ ]:
class YOLOPAFPN(nn.Module):
    def __init__(self, ch):
        super().__init__()
        c3, c4, c5 = ch
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.l5 = BaseConv(c5, c4, 1, 1); self.p4 = CSPLayer(2 * c4, c4, num_bottlenecks=1)
        self.l4 = BaseConv(c4, c3, 1, 1); self.p3 = CSPLayer(2 * c3, c3, num_bottlenecks=1)
        self.d3 = BaseConv(c3, c3, 3, 2); self.n4 = CSPLayer(2 * c3, c4, num_bottlenecks=1)
        self.d4 = BaseConv(c4, c4, 3, 2); self.n5 = CSPLayer(2 * c4, c5, num_bottlenecks=1)

    def forward(self, xT, ff=None):                        # xT: [x3,x4,x5]; ff: [f3,f4,f5] or None
        x3, x4, x5 = xT
        if ff is not None:                                 # the three ⊕ of Fig 2, at the laterals
            x3 = x3 + ff[0]; x4 = x4 + ff[1]; x5 = x5 + ff[2]
        a  = self.l5(x5)
        p4 = self.p4(torch.cat([self.up(a), x4], 1)); b = self.l4(p4)
        p3 = self.p3(torch.cat([self.up(b), x3], 1))
        n4 = self.n4(torch.cat([self.d3(p3), b], 1))
        n5 = self.n5(torch.cat([self.d4(n4), a], 1))
        return [p3, n4, n5]

In [ ]:
class MOCID(nn.Module):
    def __init__(self, num_classes=1, num_frames=5, img_size=512,
                 base_channels=32, d_state=16):                     # 64 -> 32 (YOLOX-s width)
        super().__init__()
        ch = [base_channels * 4, base_channels * 8, base_channels * 16]   # 128, 256, 512
        self.backbone = SpatioTemporalBackbone(3, base_channels, num_frames, img_size)
        self.disp     = DisplacementNet(ch, d_state)
        self.fpn      = YOLOPAFPN(ch)
        # YOLOX-s head: reference channels 256/512/1024 scaled by width=0.5 ->
        # expects 128/256/512 inputs (matches FPN) and 128-wide towers (was 256).
        self.head     = YOLOXHead(num_classes, width=0.5, in_channels=[256, 512, 1024])
        self.loss_fn  = YOLOLoss(num_classes, fp16=False, strides=[8, 16, 32])

    def forward(self, clip, labels=None, use_dam=True):
        feats = self.backbone(clip)
        if use_dam:
            f_T, f_f = self.disp(feats)
        else:
            f_T, f_f = [f[:, -1] for f in feats], None
        outs = self.head(self.fpn(f_T, f_f))
        
        if labels is not None:
            with torch.amp.autocast('cuda', enabled=False):
                outs_fp32 = [out.float() for out in outs]
                return self.loss_fn(outs_fp32, labels)
                
        return outs

In [ ]:
m = MOCID()
def pm(model, dam):
    it = model.parameters() if dam else (p for n, p in model.named_parameters()
                                         if not n.startswith("disp."))
    return sum(p.numel() for p in it) / 1e6
print(f".+FISTA (no DAM) : {pm(m, False):.2f} M   (target 9.45)")
print(f".+FISTA+DAM      : {pm(m, True):.2f} M    (target 13.05)")

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def _strip_compile(sd):
    return {k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k: v
            for k, v in sd.items()}

def save_ckpt(path, model, opt, sched, ep, stage, tag):
    torch.save({
        "model": _strip_compile(model.state_dict()),
        "opt":   opt.state_dict(),
        "sched": sched.state_dict(),
        "epoch": ep,
        "stage": stage,
        "tag":   tag,
        "rng_cpu":  torch.get_rng_state().cpu(),
        "rng_cuda": [s.cpu() for s in torch.cuda.get_rng_state_all()],
    }, path + ".tmp")
    os.replace(path + ".tmp", path)

In [ ]:
import csv, time, os, argparse, sys
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from eval import evaluate, collate_eval


def _restore_rng(ckpt):
    try:
        if "rng_cpu" in ckpt:
            torch.set_rng_state(ckpt["rng_cpu"].cpu().to(torch.uint8))
        if "rng_cuda" in ckpt:
            torch.cuda.set_rng_state_all([s.cpu().to(torch.uint8) for s in ckpt["rng_cuda"]])
    except Exception as e:
        print(f"[warn] skipping RNG restore ({e})")


def _sgd(model, cfg, epochs):
    opt = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=cfg.LR_INIT, momentum=cfg.MOMENTUM,
                          weight_decay=cfg.WEIGHT_DECAY)
    milestone = int(epochs * cfg.LR_MILESTONE)
    warmup = cfg.WARMUP_EPOCHS

    def lr_lambda(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        return 1.0 if ep < milestone else cfg.LR_DECAY

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


def log_eval(csv_path, row):
    new = not os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new: w.writeheader()
        w.writerow(row)


def _run(model, loader, val_loader, opt, sched, epochs, device, use_dam, tag, stage,
         start_ep=0, ckpt_every=5, ckpt_path="ckpt_last.pth",
         final_path=None, eval_every=20, csv_path="eval_log.csv"):

    scaler = torch.amp.GradScaler('cuda')

    for ep in range(start_ep, epochs):
        model.train()
        if use_dam:
            model.backbone.eval()
        tot = nb = 0
        pbar = tqdm(loader, desc=f"[{tag}] {ep+1}/{epochs}", leave=False, mininterval=30)

        for clip, labels in pbar:
            clip = clip.to(device, non_blocking=True)
            labels = [l.to(device, non_blocking=True) for l in labels]

            with torch.amp.autocast('cuda'):
                loss = model(clip, labels, use_dam=use_dam)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            scaler.step(opt)
            scaler.update()

            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{tot/nb:.4f}",
                             lr=f"{opt.param_groups[0]['lr']:.2e}")

        sched.step()
        avg = tot / max(nb, 1)
        print(f"[{tag}] {ep+1}/{epochs}  loss {avg:.4f}  lr {opt.param_groups[0]['lr']:.2e}")

        if (ep + 1) % ckpt_every == 0 or (ep + 1) == epochs:
            save_ckpt(ckpt_path, model, opt, sched, ep, stage, tag)

        if (ep + 1) % eval_every == 0 or (ep + 1) == epochs:
            ap50, f1 = evaluate(model, val_loader, device, use_dam,
                                strides=[8, 16, 32], num_classes=1)
            log_eval(csv_path, {"time": time.strftime("%Y-%m-%d %H:%M:%S"),
                                "stage": stage, "tag": tag, "epoch": ep + 1,
                                "use_dam": use_dam, "ap50": round(ap50 * 100, 2),
                                "f1": round(f1 * 100, 2), "avg_loss": round(avg, 4)})
            print(f"[{tag}] eval @ ep{ep+1}: AP50 {ap50*100:.2f}  F1 {f1*100:.2f}")

    if final_path is not None:
        save_ckpt(final_path, model, opt, sched, epochs - 1, stage, tag)
        print(f"[{tag}] saved stage checkpoint -> {final_path}")


def train_mocid(cfg, device="cuda", tag="default"):
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

    run_dir = os.path.join("runs", tag)
    os.makedirs(run_dir, exist_ok=True)
    p = lambda name: os.path.join(run_dir, name)
    resume, csv_path = p("ckpt_last.pth"), p("eval_log.csv")
    print(f"[run] tag={tag} -> {run_dir}")

    train_ds = MOCIDDataset(cfg.train_path, T=cfg.T, img_size=cfg.IMG_SIZE, is_train=True)
    val_ds   = MOCIDDataset(cfg.val_path,   T=cfg.T, img_size=cfg.IMG_SIZE, is_train=False)
    assert len(train_ds) > 0 and len(val_ds) > 0

    loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                        num_workers=4, collate_fn=collate_mocid, drop_last=True,
                        pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                            num_workers=4, collate_fn=collate_eval,
                            pin_memory=True, persistent_workers=True)

    model = MOCID(num_classes=1, num_frames=cfg.T, img_size=cfg.IMG_SIZE[0]).to(device)
    model = torch.compile(model)

    ckpt = torch.load(resume, map_location="cpu") if os.path.exists(resume) else None
    if ckpt:
        getattr(model, "_orig_mod", model).load_state_dict(_strip_compile(ckpt["model"]))
        _restore_rng(ckpt)
        print(f"resuming from stage {ckpt['stage']} epoch {ckpt['epoch']+1}")

    if ckpt is None or ckpt["stage"] == 1:
        for pm in model.disp.parameters(): pm.requires_grad_(False)
        opt, sched = _sgd(model, cfg, cfg.EPOCHS_SPTBACKBONE)
        start = 0
        if ckpt and ckpt["stage"] == 1:
            opt.load_state_dict(ckpt["opt"]); sched.load_state_dict(ckpt["sched"])
            start = ckpt["epoch"] + 1
        _run(model, loader, val_loader, opt, sched, cfg.EPOCHS_SPTBACKBONE, device,
             use_dam=False, tag="STB", stage=1, start_ep=start,
             ckpt_path=resume, final_path=p("ckpt_stage1.pth"),
             eval_every=cfg.EVAL_EVERY, csv_path=csv_path)
        ckpt = None

    for pm in model.backbone.parameters(): pm.requires_grad_(False)
    for pm in model.disp.parameters():     pm.requires_grad_(True)
    opt, sched = _sgd(model, cfg, cfg.EPOCHS_DAM)
    start = 0
    if ckpt and ckpt["stage"] == 2:
        opt.load_state_dict(ckpt["opt"]); sched.load_state_dict(ckpt["sched"])
        start = ckpt["epoch"] + 1
    _run(model, loader, val_loader, opt, sched, cfg.EPOCHS_DAM, device,
         use_dam=True, tag="DAM", stage=2, start_ep=start, ckpt_every=1,
         ckpt_path=resume, final_path=p("ckpt_stage2.pth"),
         eval_every=cfg.EVAL_EVERY, csv_path=csv_path)
    return model


def _get_tag():
    if "ipykernel" in sys.modules:
        return "default"
    ap = argparse.ArgumentParser()
    ap.add_argument("--tag", default="default")
    args, _ = ap.parse_known_args()
    return args.tag


if __name__ == "__main__":
    train_mocid(Config(), tag=_get_tag())